In [14]:
# setup
import anthropic        # the official Anthropic SDK
import asyncio          # lets us run the specialists concurrently
from dotenv import load_dotenv

import json             # for saving/loading the session state
import os               # use for checking if the file exists on resume

# Load variables from the .env file into the system environment
load_dotenv()

# A small, fast model is plenty here. Swap for any model your key can reach.
MODEL_NAME = "claude-haiku-4-5-20251001"

# One async client, shared by everyone. It reads ANTHROPIC_API_KEY from the env.
client = anthropic.AsyncAnthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

# Where we save progress so we can resume later.
STATE_FILE = "session_state.json"

In [17]:
# ---------------------------------------------------------------------------
# 1. THE SPECIALISTS (the spokes)
# ---------------------------------------------------------------------------
# Each specialist is just a name + a "system prompt" describing its job.
# This is a fixed set of roles, which keeps the example simple.

SPECIALISTS = {
    "market_sizing": "You are a market-sizing analyst. Estimate the size of a "
                     "market and explain your reasoning briefly.",
    "competitor_scan": "You are a competitive analyst. Identify the main "
                       "competitors in a market and summarize each briefly.",
    "patent_search": "You are a patent researcher. Describe the kinds of "
                     "patents and IP activity relevant to a topic.",
}


# ---------------------------------------------------------------------------
# 2. THE COORDINATOR: DECOMPOSE
# ---------------------------------------------------------------------------
def decompose(question):
    """
    Break the big question into one subtask per specialist.

    Returns the "session state": the original question plus a list of subtask
    records. Each record is a plain dict with four fields that EVERY other
    part of the program reads and writes:

        name   - which specialist handles it
        brief  - the exact, self-contained instructions for that specialist
        status - "pending" | "done" | "failed"
        result - the specialist's answer (empty until it finishes)

    Locking down this record shape first is what keeps the rest simple.
    """
    subtasks = []
    for name, _role in SPECIALISTS.items():
        # The brief is the specialist's ISOLATED context: it contains only
        # what that specialist needs, not the other specialists' work.
        brief = f"Regarding this question: '{question}'\n\nDo your part of the analysis."
        subtasks.append({
            "name": name,
            "brief": brief,
            "status": "pending",   # nothing has run yet
            "result": "",
        })
    return {"question": question, "subtasks": subtasks}


# ---------------------------------------------------------------------------
# 3. THE SPECIALIST RUNNER (one spoke doing its job)
# ---------------------------------------------------------------------------
async def run_specialist(subtask):
    """
    Run a single specialist and update its record IN PLACE.

    Two important design points:
      * CONTEXT ISOLATION: we build a brand-new 'messages' list here containing
        only this specialist's brief. Nothing from the other specialists leaks
        in.
      * FAILURE RECOVERY: the whole call is wrapped in try/except. If it fails,
        we mark this ONE subtask "failed" and return normally. We never let the
        error escape, so the other specialists keep running.
    """
    name = subtask["name"]
    system_prompt = SPECIALISTS[name]

    try:
        # Fresh, isolated context: just this specialist's brief.
        response = await client.messages.create(
            model=MODEL_NAME,
            max_tokens=500,
            system=system_prompt,
            messages=[{"role": "user", "content": subtask["brief"]}],
        )
        # Pull the text out of the response and store it.
        subtask["result"] = response.content[0].text
        subtask["status"] = "done"

    except Exception as e:
        subtask["result"] = f"ERROR: {e}"
        subtask["status"] = "failed"

    return subtask

# ---------------------------------------------------------------------------
# 4. THE COORDINATOR: DISPATCH (in parallel)
# ---------------------------------------------------------------------------
async def dispatch(state):

    """
    Run every subtask that isn't already done, all at the same time.

    * PARALLEL DISPATCH: asyncio.gather starts all the specialist calls at
      once, so the total time is about the SLOWEST specialist, not the sum of
      all of them.
    * On RESUME we skip subtasks that already finished, so we don't redo work.
    """
    # Only run subtasks that still need doing (status is not "done").( status = not done)
    todo = [st for st in state["subtasks"] if st["status"] != "done"]

    if not todo:
        print("Nothing to do — every subtask is already done.")
        return state
    
    print(f"Dispatching {len(todo)} specialist(s) in parallel:" f" {[st['name'] for st in todo]}")

    # Launch them all together and wait for all to finish.
    await asyncio.gather(*(run_specialist(st) for st in todo))

    # Save progress right after dispatch so a later crash can resume from here.
    save_state(state)
    return state

# ---------------------------------------------------------------------------
# 5. THE COORDINATOR: AGGREGATE
# ---------------------------------------------------------------------------
def aggregate(state):
    """
    Merge the specialists' results into one combined report.

    Failed subtasks are included too, clearly labeled, so the reader can see
    what succeeded and what didn't.
    """
    lines = [f"COMBINED REPORT FOR: {state['question']}", "=" * 60]
    for st in state["subtasks"]:
        lines.append(f"\n## {st['name']} [{st['status']}]")
        lines.append(st["result"] or "<no result>")
    return "\n".join(lines)


# ---------------------------------------------------------------------------
# 6. SESSION STATE: SAVE / LOAD (for resume)
# ---------------------------------------------------------------------------
def save_state(state):
    """Write the whole state to a JSON file. This is all 'resume' needs."""
    with open(STATE_FILE, "w") as f:
        json.dump(state, f, indent=2)

def load_state():
    """Load a saved state from disk, or return None if there isn't one."""
    if os.path.exists(STATE_FILE):
        with open(STATE_FILE) as f:
            return json.load(f)
    return None


# ---------------------------------------------------------------------------
# 7. TOP-LEVEL: run a fresh question, or resume an interrupted one
# ---------------------------------------------------------------------------
async def run(question):
    """Start a brand-new run from a question."""
    state = decompose(question)
    await dispatch(state)
    return aggregate(state)

async def resume():
    """Continue a saved run: only the unfinished subtasks get re-run."""
    state = load_state()
    if state is None:
        print("No saved session to resume.")
        return None
    print("Resuming saved session...")
    await dispatch(state)            # dispatch skips anything already "done"
    return aggregate(state)

# ---------------------------------------------------------------------------
# 8. DEMONSTRATION
# ---------------------------------------------------------------------------
async def main():
    """
    Show the whole thing end to end:

      Run 1: a normal parallel run, but we FORCE one specialist to fail so you
             can see failure recovery (the other two still finish).
      Run 2: a RESUME that re-runs only the failed subtask and completes it.
    """

    question = "What is the opportunity for a new AI note-taking app?"

    # ----- RUN 1: full run with one forced failure -----------------------
    print("\n########## RUN 1: full run (one specialist forced to fail) ##########\n")
    state = decompose(question)

    # Force a failure to prove recovery: point one specialist at a bad model
    # name by temporarily breaking its brief handling. Simplest way: mark it
    # so run_specialist raises. Here we just give it an impossible request by
    # swapping the model for that one call via a wrapper.
    async def run_one_broken(subtask):
        if subtask["name"] == "patent_search":
            # Simulate a tool/API failure for this subtask only.
            try:
                raise RuntimeError("simulated API failure")
            except Exception as error:
                subtask["result"] = f"ERROR: {error}"
                subtask["status"] = "failed"
            return subtask
        return await run_specialist(subtask)


    # Dispatch by hand for the demo so we can inject the broken one.
    todo = [st for st in state["subtasks"] if st["status"] != "done"]
    print(f"Dispatching {len(todo)} specialists in parallel...")
    await asyncio.gather(*(run_one_broken(st) for st in todo))
    save_state(state)

    print(aggregate(state))
    print("\nStatuses after run 1:",
          {st["name"]: st["status"] for st in state["subtasks"]})

    # ----- RUN 2: resume, which fixes only the failed subtask ------------
    print("\n########## RUN 2: resume (only the failed subtask re-runs) ##########\n")
    # A resume normally reloads from disk. To make the demo actually succeed,
    # reset the failed subtask to "pending" so it will be retried for real.
    resumed = load_state()
    for st in resumed["subtasks"]:
        if st["status"] == "failed":
            st["status"] = "pending"
    save_state(resumed)

    report = await resume()          # only re-runs the pending (was failed) one
    print(report)
    print("\nStatuses after resume:",
          {st["name"]: st["status"] for st in load_state()["subtasks"]})
    
await main()


########## RUN 1: full run (one specialist forced to fail) ##########

Dispatching 3 specialists in parallel...
COMBINED REPORT FOR: What is the opportunity for a new AI note-taking app?

## market_sizing [done]
# AI Note-Taking App Market Opportunity

## Market Size Estimate: **$8-12 billion annually** (by 2025)

### Reasoning:

**Total Addressable Market (TAM):**
- **Knowledge workers globally:** ~1.5 billion people
- **Willing to pay for productivity tools:** ~15-20% = 225-300M people
- **Average willingness to pay:** $40-80/year for premium note-taking
- **Base TAM:** $9-24 billion

**Serviceable Market (SAM):**
Focus on English-speaking developed markets with strong tech adoption:
- ~150M potential users × $50-70 average = **$7.5-10.5 billion**

**Current Indicators:**
- Notion valued at $10B (2021) with ~30M users
- Microsoft OneNote: massive but legacy
- Obsidian: $1M+ MRR (private)
- Growing segment spending: note-taking app revenue up 40%+ YoY

### Key Drivers for AI Note-Tak